## Multiple Chain 快速入门

Runnables 可以轻松地用来串联多个 Chains，使用 RunnablePassthrough 将输出同时传给多条后继链。

```
     Input
      / \
     /   \
 Chain1 Chain2
     \   /
      \ /
      Combine
```

本指南展示如何使用 Runnable 实现多个 AI 关于相同话题的辩论：

```
    输入话题
       |
       |
    原始观点
      / \
     /   \
 正面论述 反面论述
     \   /
      \ /
     最终总结
```

In [1]:
# 导入相关模块，包括运算符、输出解析器、聊天模板、ChatOpenAI 和 运行器
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough

# 创建一个计划器，生成一个关于给定输入的论证
planner = (
    ChatPromptTemplate.from_template("生成关于以下内容的论点: {input}")
    | ChatOpenAI()
    | StrOutputParser()
    | {"base_response": RunnablePassthrough()}
)

# 创建正面论证的处理链，列出关于基础回应的正面或有利的方面
arguments_for = (
    ChatPromptTemplate.from_template(
        "列出关于{base_response}的正面或有利的方面"
    )
    | ChatOpenAI()
    | StrOutputParser()
)

# 创建反面论证的处理链，列出关于基础回应的反面或不利的方面
arguments_against = (
    ChatPromptTemplate.from_template(
        "列出关于{base_response}的反面或不利的方面"
    )
    | ChatOpenAI()
    | StrOutputParser()
)

# 创建最终响应者，综合原始回应和正反论点生成最终的回应
final_responder = (
    ChatPromptTemplate.from_messages(
        [
            ("ai", "{original_response}"),
            ("human", "正面观点:\n{results_1}\n\n反面观点:\n{results_2}"),
            ("system", "给出批评后生成最终回应"),
        ]
    )
    | ChatOpenAI()
    | StrOutputParser()
)

# 构建完整的处理链，从生成论点到列出正反论点，再到生成最终回应
chain = (
    planner
    | {
        "results_1": arguments_for,
        "results_2": arguments_against,
        "original_response": itemgetter("base_response"),
    }
    | final_responder
)

In [2]:
chain.invoke({"input": "房地产低迷"})

'在房地产市场低迷的情况下，我们需要综合考虑正面和反面观点，采取有针对性的政策和措施来化解潜在的负面影响，同时促进市场的健康发展。可以通过以下方式应对：\n\n1. 加强监管和政策引导，防止房地产市场过度波动和泡沫化，确保市场稳定和可持续发展。\n\n2. 支持房地产企业进行结构调整和转型升级，提高行业整体竞争力和抗风险能力。\n\n3. 加大金融支持力度，确保房地产市场的融资环境稳定，避免出现资金链断裂等问题。\n\n4. 加强对购房者和投资者的引导和教育，提高风险意识和理性投资水平，避免盲目跟风和投机行为。\n\n5. 加强与相关部门的协调合作，形成多方共识，共同推动房地产市场的健康发展和经济稳定。通过综合施策，我们有信心应对房地产市场低迷带来的挑战，实现市场的良性循环和可持续发展。'

#### 流式输出

In [3]:
## chain 最终输出经过了 StrOutputParser 处理，所以可以直接输出流式输出 s
for s in chain.stream({"input": "全球经济"}):
    print(s, end="", flush=True)

全球经济的发展确实存在着一些负面影响和挑战，如不平等现象加剧、资源过度开发、贸易保护主义等问题。这些挑战需要各国政府和国际组织共同努力合作，制定更加公平和可持续的政策，解决全球经济发展中的不平等和不稳定问题，实现可持续和共同繁荣。通过加强国际合作、促进可持续发展、推动技术创新等措施，可以更好地应对全球经济发展中的挑战，实现经济增长和社会进步的双赢局面。因此，我们应该正视全球经济发展中存在的问题，并积极寻求解决之道，共同推动全球经济走向更加稳定和可持续的发展道路。

### Homework: 实现一个多链版本的代码生成，输入功能需求，输出2种（Python，Java）以上编程语言的代码实现。

In [6]:
# 导入相关模块，包括运算符、输出解析器、聊天模板、ChatOpenAI 和 运行器
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough


# 创建一个计划器，生成一个关于给定输入的解释
planner = (
    ChatPromptTemplate.from_template("生成关于这个算法的描述信息: {input}")
    | ChatOpenAI()
    | StrOutputParser()
    | {"base_response": RunnablePassthrough()}
)

# 创建正面论证的处理链，列出关于基础回应的正面或有利的方面
arguments_for = (
    ChatPromptTemplate.from_template(
        "列出关于{base_response}的Java代码实现"
    )
    | ChatOpenAI()
    | StrOutputParser()
)

# 创建反面论证的处理链，列出关于基础回应的反面或不利的方面
arguments_against = (
    ChatPromptTemplate.from_template(
        "列出关于{base_response}的Python代码实现"
    )
    | ChatOpenAI()
    | StrOutputParser()
)

# 创建最终响应者，综合原始回应和正反论点生成最终的回应
final_responder = (
    ChatPromptTemplate.from_messages(
        [
            ("ai", "{original_response}"),
            ("human", "Java代码:\n{results_1}\n\nPython代码:\n{results_2}"),
            ("system", "列出Java代码和Python代码，并给出最终的回应"),
        ]
    )
    | ChatOpenAI()
    | StrOutputParser()
)

# 构建完整的处理链，从生成论点到列出正反论点，再到生成最终回应
chain = (
    planner
    | {
        "results_1": arguments_for,
        "results_2": arguments_against,
        "original_response": itemgetter("base_response"),
    }
    | final_responder
)

In [7]:
chain.invoke({"input": "快速排序"})

'Java代码:\n```java\npublic class QuickSort {\n\n    public static void quickSort(int[] arr, int left, int right) {\n        if (left < right) {\n            int pivot = partition(arr, left, right);\n            quickSort(arr, left, pivot - 1);\n            quickSort(arr, pivot + 1, right);\n        }\n    }\n\n    public static int partition(int[] arr, int left, int right) {\n        int pivot = arr[right];\n        int i = left - 1;\n        for (int j = left; j < right; j++) {\n            if (arr[j] < pivot) {\n                i++;\n                int temp = arr[i];\n                arr[i] = arr[j];\n                arr[j] = temp;\n            }\n        }\n        int temp = arr[i + 1];\n        arr[i + 1] = arr[right];\n        arr[right] = temp;\n        return i + 1;\n    }\n\n    public static void main(String[] args) {\n        int[] arr = {5, 2, 9, 3, 7, 6, 8};\n        quickSort(arr, 0, arr.length - 1);\n        for (int num : arr) {\n            System.out.print(num + " ");